In [2]:
import numpy as np
import json
import fastai
import torch
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pprint import pprint
from pathlib import Path
from omegaconf import OmegaConf

from fastai.text.all import *

from GrooveModel import Datasets, Tokenizers, CollateFunctions
from GrooveModel.Utils import DNAValue


In [3]:
# Paths
BASE_PATH = Path.cwd().parent
DATA_PATH = BASE_PATH / 'Data'
DNA_PATH = DATA_PATH / 'dnas.json'
MODEL_PATH = BASE_PATH / 'GrooveModel'
XLSTM_PATH = MODEL_PATH / 'xlstm'

In [4]:
ds_config_string = f"""
dna_path: {DNA_PATH}
# ADD more params like subset, training/validation/testing,...
convert_to_tensor: True
"""

# Final OmegaConf config
ds_conf = OmegaConf.create(ds_config_string)


In [5]:
# Dataset
dna_ds = Datasets.DNANextTokenDataset(ds_conf, 'train', tokenizer=Tokenizers.MultiDimDNATokenizer)
len(dna_ds)

622

In [6]:
dna_ds[1]

(tensor([[  1, 127,   0,  ...,   0,  16, 480],
         [  6, 127,   0,  ...,   0,  16, 480],
         [  0,   0,   1,  ...,   0,  16, 480],
         ...,
         [  3, 127,  11,  ...,   0,  16, 480],
         [  1, 127,  12,  ...,   0,  16, 480],
         [  0,   0,  13,  ...,   0,  16, 480]], dtype=torch.uint16),
 tensor([[  6, 127,   0,  ...,   0,  16, 480],
         [  0,   0,   1,  ...,   0,  16, 480],
         [  0,   0,   2,  ...,   0,  16, 480],
         ...,
         [  1, 127,  12,  ...,   0,  16, 480],
         [  0,   0,  13,  ...,   0,  16, 480],
         [  1, 127,  14,  ...,   0,  16, 480]], dtype=torch.uint16))

In [7]:
Tokenizers.DNAToken.from_tensor(dna_ds[0][0][0])

DNAToken(Instrument=2, Velocity=127, BeatUnit=0, BeatUnitOffset=480, GridFactor=2, Bpm=160, TimeSignature=0, NumberOfBars=3, TicksPerQuarter=480)

In [10]:
# DATALOADER

dataloader = torch.utils.data.DataLoader(dna_ds, batch_size=2, shuffle=True, collate_fn=CollateFunctions.dna_collate_fn)

# Pad zeroes at end:
# The most powerful attribute of LSTMs and RNNs in general is that their parameters are shared along the time frames(Parameters recur over time frames) but the parameter sharing relies upon the assumption that the same parameters can be used for different time steps i.e. the relationship between the previous time step and the next time step does not depend on t as explained here in page 388, 2nd paragraph.
#
# In short, padding zeros at the end, theoretically should not change the accuracy of the model. I used the adverb theoretically because at each time step LSTM's decision depends on its cell state among other factors and this cell state is kind of a short summary of the past frames. As far as I understood, that past frames may be missing in your case. I think what you have here is a little trade-off.
#
# I would rather pad zeros at the end because it doesn't completely conflict with the underlying assumption of RNNs and it's more convenient to implement and keep track of.
#
# On the implementation side, I know tensorflow calculates the loss function once you give it the sequences and the actual sequence size of each sample(e.g. for 4 5 6 7 0 0 0 0 0 0 you also need to give it the actual size which is 4 here) assuming you're implementing the option 2. I don't know whether there is an implementation for option 1, though.
padded_inputs, padded_targets, packed_inputs, lengths = next(iter(dataloader))
pprint(padded_inputs.shape)
pprint(padded_targets.shape)
pprint(packed_inputs)
pprint(lengths)

# TODO: HANDLE PACKED PADDED inputs/targets
# what is packing?
# check how the model handles padding
# rewrite embedding/pos encoding

torch.Size([2, 294, 9])
torch.Size([2, 294, 9])
PackedSequence(data=tensor([[  1, 127,   0,  ...,   0,  16, 480],
        [  1, 127,   0,  ...,   0,   5, 480],
        [  6, 127,   0,  ...,   0,  16, 480],
        ...,
        [  2, 127,  11,  ...,   0,  16, 480],
        [  2, 127,  12,  ...,   0,  16, 480],
        [  0,   0,  13,  ...,   0,  16, 480]], dtype=torch.uint16), batch_sizes=tensor([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [14]:
from GrooveModel.Utils.DNAGridFactor import GRID_FACTORS_SIZE
from GrooveModel.Utils.TimeSignatures import TIME_SIGNATURES_SIZE
from GrooveModel.Utils.DNAValue import DNA_VALUE_SIZE
from GrooveModel.Utils.DNAVelocity import VELOCITY_SIZE
from GrooveModel.Utils.DNAOffset import OFFSET_TICKS_SIZE


#TODO: BPM normalization (will do 0 - 300 for right now)
BPM_SIZE = 300

# vocab sizes
instruments_vocab_size = DNA_VALUE_SIZE
velocities_vocab_size = VELOCITY_SIZE
offsets_vocab_size = OFFSET_TICKS_SIZE
bpm_vocab_size = BPM_SIZE
time_signature_vocab_size = TIME_SIGNATURES_SIZE
grid_factor_vocab_size = GRID_FACTORS_SIZE

combined_vocab_size = DNA_VALUE_SIZE + VELOCITY_SIZE + OFFSET_TICKS_SIZE + GRID_FACTORS_SIZE + BPM_SIZE + TIME_SIGNATURES_SIZE + GRID_FACTORS_SIZE

# embedding dimensions
instrument_embedding_dim = 4 # 7 values
velocity_embedding_dim = 16 # 128 values
offset_embedding_dim = 32 # 960 values
time_signature_embedding_dim = 8 # 40 values
grid_embedding_dim = 4 # 5 values
bpm_embedding_dim = 16 # 300 values
position_embedding_dim = 16 # positional embedding is concatenated

embedding_config_string = f"""
embeddings:
    instruments:
      vocab_size: {instruments_vocab_size}
      embedding_dim: {instrument_embedding_dim}
    velocities:
      vocab_size: {velocities_vocab_size}
      embedding_dim: {velocity_embedding_dim}
    offsets:
      vocab_size: {offsets_vocab_size}
      embedding_dim: {offset_embedding_dim}
    time_signature:
      vocab_size: {time_signature_vocab_size}
      embedding_dim: {time_signature_embedding_dim}
    grid_factor:
      vocab_size: {grid_factor_vocab_size}
      embedding_dim: {grid_embedding_dim}
    bpm:
      vocab_size: {bpm_vocab_size}
      embedding_dim: {bpm_embedding_dim}
"""

# Final OmegaConf config
embedding_config = OmegaConf.create(embedding_config_string)
embedding_config

{'embeddings': {'instruments': {'vocab_size': 64, 'embedding_dim': 4}, 'velocities': {'vocab_size': 128, 'embedding_dim': 16}, 'offsets': {'vocab_size': 960, 'embedding_dim': 32}, 'time_signature': {'vocab_size': 12, 'embedding_dim': 8}, 'grid_factor': {'vocab_size': 17, 'embedding_dim': 4}, 'bpm': {'vocab_size': 300, 'embedding_dim': 16}}, 'positional_embedding_dim': 16, 'total_embedding_dim': 96}

In [15]:
from omegaconf import DictConfig


# TODO Rewrite for batching


In [16]:
# TODO Rewrite for batching


In [17]:
# TODO Rewrite for batching


In [18]:
embedder = DNAEmbedding(embedding_config)
embedder

DNAEmbedding(
  (content_embedding): DNAContentEmbedding(
    (instrument_embedding): Embedding(64, 4)
    (velocity_embedding): Embedding(128, 16)
    (offset_embedding): Embedding(960, 32)
    (time_signature_embedding): Embedding(12, 8)
    (grid_embedding): Embedding(17, 4)
    (bpm_embedding): Embedding(300, 16)
  )
  (positional_encoding): SinusoidalPositionalEncoding()
)

In [19]:
embedder.content_embedding.instrument_embedding.weight

Parameter containing:
tensor([[-0.7947, -0.5072, -1.8135,  1.2095],
        [-0.5719, -1.8196,  0.1877,  0.0083],
        [ 2.6041,  0.3421, -1.0696, -0.4745],
        [ 0.1919, -0.0537,  1.6957,  0.9429],
        [-0.7705, -1.0214,  0.3898,  0.8477],
        [ 0.7861, -0.8275, -1.4162, -0.1327],
        [ 0.3986, -0.4810, -1.6315, -0.3216],
        [-0.5830, -0.9617,  0.1647,  0.0440],
        [-0.6154,  0.7494,  1.6108, -0.9611],
        [-0.1511, -1.0808, -0.7889,  0.8642],
        [-0.6748,  1.6061, -1.8007,  0.4087],
        [-0.0878,  0.4831, -1.4501,  1.1264],
        [ 0.4057,  0.1799, -1.5462, -1.3151],
        [ 0.0521,  0.1601, -0.8599, -0.1488],
        [-3.1448,  0.3800,  0.0480,  1.5247],
        [ 1.8505, -0.8563, -0.4170, -0.3708],
        [-0.1958,  0.7100, -0.2124, -0.0572],
        [-0.8525,  0.2991,  0.0428,  0.1486],
        [-0.5066, -1.4159,  1.6480,  0.2465],
        [ 1.0407, -0.8490,  0.8975,  0.7630],
        [-0.4446,  0.6802, -2.0668, -0.7534],
        [ 0.

In [20]:
try:
    for token in all_tokens:
        embedder(token)

except Exception as e:
    print(token)
    print(e.with_traceback())

NameError: name 'token' is not defined

In [91]:
embedder(all_tokens[1])

tensor([ 1.4013e+00,  3.1400e+00,  1.8500e+00, -1.9692e+00, -3.7603e-01,
         6.4551e-01, -6.1554e-01, -6.7255e-01, -1.4563e+00,  8.8162e-01,
        -2.7080e+00, -1.4758e+00,  4.5351e-01, -2.0650e+00, -7.0640e-01,
        -9.0175e-01,  4.5362e-01, -2.6467e-01, -7.3718e-01,  1.8659e-01,
        -6.7350e-01, -1.8315e+00,  3.9333e-01, -1.5081e-01,  9.9881e-01,
         7.3192e-01, -1.1077e+00, -1.9708e-01,  8.9147e-01, -3.4369e-01,
         1.3448e+00,  3.4643e-02, -4.8758e-01, -8.1347e-02, -4.4385e-01,
         2.7023e+00, -1.6174e+00,  6.8890e-01, -7.0169e-01, -1.1170e+00,
        -4.7950e-01,  1.7316e+00, -5.0612e-01,  1.0467e+00,  5.2830e-01,
        -9.3637e-01, -1.7109e+00,  2.8165e-01,  5.6732e-02,  1.4650e-01,
         1.3814e+00,  1.1462e+00, -4.0080e-01, -4.2155e-01, -2.0276e+00,
         5.8164e-01,  6.9316e-01, -9.0715e-01, -3.7214e-01, -1.0331e+00,
        -7.0130e-01, -1.4616e+00, -7.1605e-01,  9.0924e-01,  1.7688e-01,
         3.7176e-01,  1.2298e+00, -7.0314e-01,  1.9

In [92]:
from GrooveModel.xlstm.xlstm.components.init import small_init_init_
from GrooveModel.xlstm.xlstm.utils import WeightDecayOptimGroupMixin
from GrooveModel.xlstm.xlstm import xLSTMBlockStackConfig, xLSTMBlockStack, xLSTMLMModelConfig
from omegaconf import OmegaConf
from pprint import pprint
from dacite import from_dict
from dacite import Config as DaciteConfig
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

class xLSTMDNAModel(WeightDecayOptimGroupMixin, nn.Module):
    config_class = xLSTMLMModelConfig

    def __init__(self, config: xLSTMLMModelConfig, **kwargs):
        super().__init__()
        self.config = config

        self.xlstm_block_stack = xLSTMBlockStack(config=config)
        self.token_embedding = DNAEmbedding(config)
        self.emb_dropout = nn.Dropout(config.dropout) if config.add_embedding_dropout else nn.Identity()

        self.lm_head = nn.Linear(
            in_features=config.embedding_dim,
            out_features=combined_vocab_size,
            bias=False,
        )

        # same weights for input and output: https://arxiv.org/abs/1608.05859
        # will skip as i have separate embedding spaces
        # if config.tie_weights:
        #     self.lm_head.weight = self.token_embedding.weight

    def reset_parameters(self):
        self.xlstm_block_stack.reset_parameters()

        small_init_init_(self.token_embedding.weight, dim=self.config.embedding_dim)

        if not self.config.tie_weights:
             small_init_init_(self.lm_head.weight, dim=self.config.embedding_dim)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        x = self.token_embedding(idx)
        x = self.emb_dropout(x)
        x = self.xlstm_block_stack(x)
        logits = self.lm_head(x)
        return logits

    def step(
        self, idx: torch.Tensor, state: dict[str, dict[str, tuple[torch.Tensor, ...]]] = None, **kwargs
    ) -> tuple[torch.Tensor, dict[str, dict[str, tuple[torch.Tensor, ...]]]]:
        x = self.token_embedding(idx)
        x = self.emb_dropout(x)
        x, state = self.xlstm_block_stack.step(x, state=state, **kwargs)
        logits = self.lm_head(x)
        return logits, state

    # TODO make compatible with own embedding
    def _create_weight_decay_optim_groups(self, **kwargs) -> tuple[Sequence[nn.Parameter], Sequence[nn.Parameter]]:
        weight_decay, no_weight_decay = super()._create_weight_decay_optim_groups(**kwargs)
        # remove token embedding and add it to the correct group, accrording to the config
        weight_decay = list(weight_decay)
        removed = 0
        for idx in range(len(weight_decay)):
            if weight_decay[idx - removed] is self.token_embedding.weight:
                weight_decay.pop(idx - removed)
                removed += 1
        weight_decay = tuple(weight_decay)
        if self.config.weight_decay_on_embedding:
            weight_decay += (self.token_embedding.weight,)
        else:
            no_weight_decay += (self.token_embedding.weight,)

        return weight_decay, no_weight_decay

In [94]:
context_length = 100
num_heads = embedder.total_embedding_dim // 16

xlstm_cfg = f"""
mlstm_block:
  mlstm:
    conv1d_kernel_size: 4
    qkv_proj_blocksize: 4
    num_heads: 4
slstm_block:
  slstm:
    backend: {'cuda' if torch.cuda.is_available() else 'vanilla'} #! only vanilla here works
    num_heads: {num_heads}
    conv1d_kernel_size: 4
    bias_init: powerlaw_blockdependent
  feedforward:
    proj_factor: 1.3
    act_fn: gelu
context_length: {context_length}
num_blocks: 7
slstm_at: [1] #[1] # for [] it also works, so if no sLSTM is in the stack
"""

cfg = OmegaConf.create(xlstm_cfg)
cfg = from_dict(data_class=xLSTMBlockStackConfig, data=OmegaConf.to_container(cfg), config=DaciteConfig(strict=True))
pprint(cfg)

xLSTMBlockStackConfig(mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(proj_factor=2.0,
                                                                          round_proj_up_dim_up=True,
                                                                          round_proj_up_to_multiple_of=64,
                                                                          _proj_up_dim=256,
                                                                          conv1d_kernel_size=4,
                                                                          qkv_proj_blocksize=4,
                                                                          num_heads=4,
                                                                          embedding_dim=128,
                                                                          bias=False,
                                                                          dropout=0.0,
                                                                

In [115]:
first_tokens = all_tokens[:context_length]

x = torch.zeros(1, context_length, embedder.total_embedding_dim)

for i, token in enumerate(first_tokens):
    # Get token embedding (returns shape: (total_embedding_dim,))
    embedding = embedder(token)

    # Insert embedding into tensor x
    x[0, i] = embedding


x.size()

torch.Size([1, 100, 96])

In [21]:
# split beat/fill ? // OPEN
# create embeddings for all required dna values // DONE
# conat them // DONE
# positional embedding according to gridunit (beat position) // DONE
# feed to xlstm
# split output tensor
# reconstruct dna unit

# Rewrite all into python files




In [ ]:
# DONT FORGET EMBEDDING NORMALIZATION
# MAYBE POOLING OF EMBEDDINGS?

1. Data Preprocessing will require a step of preforming the file read layout (mid + text)
2. Make a class for preparing midi files and their meta info to dna-layout
3. Make class abstract and define specific readers for each source (foundational vs. lmd vs. basti ,...)
4. Run json extractor
5. Create dataset with json
6. create tokens
7. embed tokens
8. ...


# DOCUMENT AFTER TEST!!!!!!!!!!!